# Finance LLM Fine-Tuning (QLoRA)

This notebook does three things, end to end, on a **free Colab T4 GPU**:

1. **Downloads a small open-source LLM locally** from Hugging Face (`Llama-3.2-3B-Instruct`, the official Meta model) and runs it for inference.
2. **Loads a finance-specific instruction dataset** (`gbharti/finance-alpaca`) from Hugging Face.
3. **Fine-tunes the model with QLoRA (4-bit + LoRA)** so it gets better at answering finance questions, then lets you chat with the fine-tuned model and (optionally) save/push it.

> **Before you start:**
> 1. In Colab: `Runtime -> Change runtime type -> T4 GPU` (or better).
> 2. On Hugging Face: visit https://huggingface.co/meta-llama/Llama-3.2-3B-Instruct and click "Agree and access repository" (free, near-instant approval), then grab a token from https://huggingface.co/settings/tokens.

**Model choice note:** `meta-llama/Llama-3.2-3B-Instruct` is Meta's official 3B model — real Llama architecture, small enough to fine-tune comfortably on a free T4 (15GB VRAM) within a single session. It's **gated**, meaning you need the one-time (usually instant) license acceptance above before downloading. If access approval is ever delayed during a live session, `MODEL_ID` can be swapped for `unsloth/Llama-3.2-3B-Instruct` in the config cell below — an ungated mirror with identical weights — as a zero-friction fallback.


## 1. Install dependencies

In [ ]:
!pip install -q -U transformers accelerate peft bitsandbytes datasets trl huggingface_hub sentencepiece
print("Done installing.")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 48.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 863.2/863.2 kB 32.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 770.3/770.3 kB 46.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 16.0 MB/s eta 0:00:00
Done installing.


## 2. Hugging Face login — **required** for the official `meta-llama/Llama-3.2-3B-Instruct`

Before running this cell:
1. Visit https://huggingface.co/meta-llama/Llama-3.2-3B-Instruct and click **"Agree and access repository"** (free, usually approved instantly).
2. Create a token at https://huggingface.co/settings/tokens (a **read** token is enough).
3. Run the cell below and paste that token when prompted.

If you skip this and `MODEL_ID` is still set to the gated `meta-llama/...` model, the download in Step 4 will fail with a 401/403 error.


In [ ]:
from huggingface_hub import notebook_login

notebook_login()   # paste your HF token here (needs read access + accepted Llama license)


## 3. Config — pick your base model & hyperparameters

In [ ]:
import torch

# --- Model ---
MODEL_ID = "meta-llama/Llama-3.2-3B-Instruct"   # OFFICIAL Meta model. GATED — see step 2 below (usually instant approval).
# Fallback if gating/approval isn't done yet or fails during the session:
# MODEL_ID = "unsloth/Llama-3.2-3B-Instruct"      # ungated mirror, identical weights/architecture, no approval needed
# Other options:
# MODEL_ID = "NousResearch/Hermes-3-Llama-3.2-3B" # ungated, ChatML format, Nous fine-tune
# MODEL_ID = "TinyLlama/TinyLlama-1.1B-Chat-v1.0" # fully open, no gating
# MODEL_ID = "microsoft/Phi-3-mini-4k-instruct"
# MODEL_ID = "mistralai/Mistral-7B-Instruct-v0.3"

# --- Dataset ---
DATASET_ID = "gbharti/finance-alpaca"   # instruction/input/output finance QA dataset

# --- Output ---
NEW_MODEL_NAME = "tinyllama-finance-qlora"
OUTPUT_DIR = "/content/finetuned-model"

# --- Training hyperparameters ---
NUM_TRAIN_EPOCHS = 1
PER_DEVICE_BATCH_SIZE = 4
GRAD_ACCUM_STEPS = 4
LEARNING_RATE = 2e-4
MAX_SEQ_LENGTH = 512
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)
if device == "cpu":
    print("WARNING: No GPU detected. Go to Runtime -> Change runtime type -> GPU.")


Using device: cuda


## 4. Download the base model locally & run it (before fine-tuning)

This downloads the model + tokenizer weights from Hugging Face to the Colab disk/cache and loads them into memory in **4-bit** (via `bitsandbytes`) so it fits comfortably on 15 GB VRAM of T4.


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
)
base_model.config.use_cache = False
base_model.config.pretraining_tp = 1

print("Model loaded locally:", MODEL_ID)
print(base_model)


config.json:   0%|          | 0.00/878 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/54.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/20.9k [00:00<?, ?B/s]

...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

Model loaded locally: meta-llama/Llama-3.2-3B-Instruct
LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 3072)
    (layers): ModuleList(
      (0-27): 28 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear4bit(in_features=3072, out_features=3072, bias=False)
          (k_proj): Linear4bit(in_features=3072, out_features=1024, bias=False)
          (v_proj): Linear4bit(in_features=3072, out_features=1024, bias=False)
          (o_proj): Linear4bit(in_features=3072, out_features=3072, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear4bit(in_features=3072, out_features=8192, bias=False)
          (up_proj): Linear4bit(in_features=3072, out_features=8192, bias=False)
          (down_proj): Linear4bit(in_features=8192, out_features=3072, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((3072,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((

In [ ]:
# Quick sanity-check inference with the BASE (not-yet-fine-tuned) model
# Uses Llama-3.2's native chat template for best-quality prompting
def ask(model, question, max_new_tokens=512):
    messages = [{"role": "user", "content": question}]

    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    ).to(model.device)

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id,
        )

    new_tokens = out[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

print(ask(base_model, "What is the difference between a stock and a bond?"))

A stock and a bond are two types of investments that represent ownership or debt in a company. The main difference between them lies in the way they give you access to the company's financials and potential for returns.

**Stocks:**

* Represent ownership in a company
* Give you a claim on a portion of the company's assets and profits
* Can be bought and sold on public stock exchanges like the New York Stock Exchange (NYSE)
* Potential for long-term capital appreciation (increase in value)
* Typically offer dividends, which are portions of the company's profits distributed to shareholders
* Risk: Stocks can be volatile, and their value can fluctuate rapidly

**Bonds:**

* Represent debt obligations of a company or government
* Typically have a fixed interest rate and maturity date
* Offer a regular income stream in the form of interest payments
* Can be bought and sold on public bond markets
* Risk: Bonds offer a relatively lower risk compared to stocks, but the return may be lower

Ke

## 5. Load the finance-specific dataset

`gbharti/finance-alpaca` is an Alpaca-style instruction dataset focused on finance/investing Q&A, with `instruction`, `input` (optional context), and `output` fields.

### What is Stanford Alpaca?

**Stanford Alpaca** is an instruction-tuned version of Meta's LLaMA language model, released by Stanford University in 2023. It was created by fine-tuning the base LLaMA model on approximately **52,000 instruction-response examples** generated using OpenAI's **text-davinci-003** model. The objective was to train the model to better understand and follow natural language instructions.

## Alpaca Dataset Format

The Stanford Alpaca dataset consists of three fields:

- **instruction** – The task or question that the user wants the model to perform.
- **input** – Optional additional context or information needed to complete the task.
- **output** – The expected response that the model should generate.

An example of an Alpaca-style dataset entry is shown below:

```json
{
  "instruction": "Summarize the article.",
  "input": "The article discusses climate change...",
  "output": "The article explains that..."
}
```

## What is an Alpaca-style Dataset?

An **Alpaca-style dataset** is any instruction-following dataset that follows the same three-field structure (`instruction`, `input`, and `output`). This format is widely used for **supervised fine-tuning (SFT)** of large language models because it teaches the model how to generate appropriate responses to user instructions.


In [ ]:
from datasets import load_dataset

raw_dataset = load_dataset(DATASET_ID, split="train")
print(raw_dataset)
print(raw_dataset[0])


README.md:   0%|          | 0.00/831 [00:00<?, ?B/s]

Cleaned_date.json: reconstructing file:   0%|          |  0.00B / 42.9MB            

Cleaned_date.json: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/68912 [00:00<?, ? examples/s]

Dataset({
    features: ['instruction', 'input', 'output', 'text'],
    num_rows: 68912
})
{'instruction': 'For a car, what scams can be plotted with 0% financing vs rebate?', 'input': '', 'output': "The car deal makes money 3 ways. If you pay in one lump payment. If the payment is greater than what they paid for the car, plus their expenses, they make a profit. They loan you the money. You make payments over months or years, if the total amount you pay is greater than what they paid for the car, plus their expenses, plus their finance expenses they make money. Of course the money takes years to come in, or they sell your loan to another business to get the money faster but in a smaller amount. You trade in a car and they sell it at a profit. Of course that new transaction could be a lump sum or a loan on the used car... They or course make money if you bring the car back for maintenance, or you buy lots of expensive dealer options. Some dealers wave two deals in front of you: get a 0%

In [ ]:
# Optional: use a subset to keep training fast on free Colab. Remove/adjust for full runs.
#To fine-tune on the entire dataset (~68,000 examples), set SUBSET_SIZE = None or remove the subset selection block below.
SUBSET_SIZE = 5000
if SUBSET_SIZE and len(raw_dataset) > SUBSET_SIZE:
    raw_dataset = raw_dataset.shuffle(seed=42).select(range(SUBSET_SIZE))

def format_example(example):
    instruction = example["instruction"]
    input_text = example.get("input", "") or ""
    output_text = example["output"]

    if input_text.strip():
        prompt = f"### Instruction:\n{instruction}\n\n### Input:\n{input_text}\n\n### Response:\n{output_text}"
    else:
        prompt = f"### Instruction:\n{instruction}\n\n### Response:\n{output_text}"
    return {"text": prompt}

formatted_dataset = raw_dataset.map(format_example, remove_columns=raw_dataset.column_names)
formatted_dataset = formatted_dataset.train_test_split(test_size=0.05, seed=42)
print(formatted_dataset)
print(formatted_dataset["train"][0]["text"][:500])


Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 4750
    })
    test: Dataset({
        features: ['text'],
        num_rows: 250
    })
})
### Instruction:
Home Renovations are expensive.. Should I only pay cash for them?

### Response:
I know that both Lowes and Home Depot (in Canada at least) will offer a 6 month deferred interest payment on all purchases over a certain dollar amount (IIRC, $500+), and sometimes run product specific 1 year deferred interest specials. This is a very effective way of financing renovations. Details: You've probably seen deferred interest -- It's very commonly used in furniture sales (No money down!!


## 6. Set up LoRA (parameter-efficient fine-tuning)

We only train a small set of low-rank adapter weights on top of the frozen, 4-bit base model — this is what makes fine-tuning feasible on a free-tier GPU.


In [ ]:
from peft import LoraConfig, prepare_model_for_kbit_training, get_peft_model

base_model = prepare_model_for_kbit_training(base_model)

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)

peft_model = get_peft_model(base_model, lora_config)
peft_model.print_trainable_parameters()


trainable params: 24,313,856 || all params: 3,237,063,680 || trainable%: 0.7511


## 7. Fine-tune with `trl`'s `SFTTrainer`

In [ ]:
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_TRAIN_EPOCHS,
    per_device_train_batch_size=PER_DEVICE_BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    optim="paged_adamw_8bit",
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    logging_steps=20,
    save_strategy="epoch",
    eval_strategy="epoch",
    bf16=True,
    max_length=MAX_SEQ_LENGTH,
    dataset_text_field="text",
    report_to="none",
)

trainer = SFTTrainer(
    model=peft_model,
    args=training_args,
    train_dataset=formatted_dataset["train"],
    eval_dataset=formatted_dataset["test"],
)

trainer.train()


[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Epoch,Training Loss,Validation Loss


Epoch,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
1,2.021989,1.839486,1.862974,564681.000000,0.571363


TrainOutput(global_step=297, training_loss=1.9921648173219828, metrics={'train_runtime': 11978.9148, 'train_samples_per_second': 0.397, 'train_steps_per_second': 0.025, 'total_flos': 1.895365446612173e+16, 'train_loss': 1.9921648173219828, 'epoch': 1.0})

## 8. Save the fine-tuned LoRA adapter (and merged model)

In [ ]:
FINAL_ADAPTER_DIR = f"{OUTPUT_DIR}/final-adapter"
trainer.model.save_pretrained(FINAL_ADAPTER_DIR)
tokenizer.save_pretrained(FINAL_ADAPTER_DIR)
print("LoRA adapter saved to:", FINAL_ADAPTER_DIR)


LoRA adapter saved to: /content/finetuned-model/final-adapter


## 9. Test the fine-tuned model on finance questions

In [ ]:
!pip install -U torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 85.8 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

# Base model used for fine-tuning
MODEL_ID = "meta-llama/Llama-3.2-3B-Instruct"   # <-- replace with your actual model

# Directory where the LoRA adapter was saved
FINAL_ADAPTER_DIR = "/content/finetuned-model/final-adapter"

# Directory to save merged model
MERGED_DIR = "/content/finetuned-model/merged-model"

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

# Load base model in BF16/FP16 (NOT 4-bit)
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.bfloat16,      # use torch.float16 if BF16 isn't supported
    device_map="auto",
)

# Load LoRA adapter
peft_model = PeftModel.from_pretrained(
    base_model,
    FINAL_ADAPTER_DIR,
)

# Merge LoRA weights into the base model
merged_model = peft_model.merge_and_unload()

# Save merged model
merged_model.save_pretrained(MERGED_DIR)
tokenizer.save_pretrained(MERGED_DIR)

print("Merged model saved to:", MERGED_DIR)

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Merged model saved to: /content/finetuned-model/merged-model


In [ ]:
print(type(merged_model))

<class 'transformers.models.llama.modeling_llama.LlamaForCausalLM'>


In [7]:
import torch

def ask(model, question, max_new_tokens=1024):
    messages = [{"role": "user", "content": question}]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt",
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    generated = outputs[0][inputs["input_ids"].shape[1]:]

    return tokenizer.decode(
        generated,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )

In [8]:
test_questions = [
    "What is diversification and why does it matter for a portfolio?",
    "Explain the difference between an ETF and a mutual fund.",
    "What factors affect a company's price-to-earnings ratio?",
]

for q in test_questions:
    print("=" * 80)
    print("Q:", q)
    print("A:", ask(merged_model, q))

Q: What is diversification and why does it matter for a portfolio?
A: Diversification is the process of spreading investments across different asset classes, sectors, geographic regions, and other factors to reduce risk and increase potential returns. The goal of diversification is to create a portfolio that is less susceptible to market fluctuations, as individual investments may perform differently under different market conditions.

Diversification is important for a portfolio because it can:

1. Reduce risk: By spreading investments across different asset classes and sectors, a portfolio can reduce the risk of losses if one investment performs poorly.
2. Increase potential returns: Diversification can also increase the potential returns of a portfolio, as individual investments may perform differently in different market conditions.
3. Improve liquidity: A diversified portfolio can be more liquid than a portfolio with a single investment, as it is less dependent on a single investm

## Notes & tips

- **Free T4 GPU (15GB VRAM):** works well for the 1.1B TinyLlama setup above. For 7B-class models (Mistral, Llama-3.1-8B), use Colab Pro with an A100, or reduce `MAX_SEQ_LENGTH` / batch size further.
- **Dataset swap:** other good finance datasets on the Hub include `FinGPT/fingpt-sentiment-train`, `causal-lm/finance`, and `AdaptLLM/finance-tasks`. Just change `DATASET_ID` and adjust `format_example` to match the dataset's columns.
- **Faster iteration:** lower `SUBSET_SIZE` and `NUM_TRAIN_EPOCHS` first to make sure the whole pipeline runs end-to-end, then scale up.
- **Out of memory?** Reduce `PER_DEVICE_BATCH_SIZE`, increase `GRAD_ACCUM_STEPS` to compensate, or reduce `MAX_SEQ_LENGTH`
